# Phase 2: Compression Analysis
Setup imports, env loading, and OpenAI async client initialization.

In [4]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv()
client = AsyncOpenAI()

In [5]:
text_a = 'The patient has chest discomfort with shortness of breath.'
text_b = 'The person reports breathing trouble and chest pain.'

async def get_embeddings():
    resp = await client.embeddings.create(
        model='text-embedding-3-large',
        input=[text_a, text_b],
    )
    return np.array(resp.data[0].embedding, dtype=np.float32), np.array(resp.data[1].embedding, dtype=np.float32)

vec_a, vec_b = await get_embeddings()
vec_a.shape, vec_b.shape

((3072,), (3072,))

In [6]:
from app.services.compression_service import matryoshka_truncate

a_m = matryoshka_truncate(vec_a, 256)
b_m = matryoshka_truncate(vec_b, 256)
cos_m = float(np.dot(a_m, b_m) / (np.linalg.norm(a_m) * np.linalg.norm(b_m)))
mem_orig = 3072 * 4
mem_mat = 256 * 4
ratio_mat = mem_orig / mem_mat
cos_m, mem_orig, mem_mat, ratio_mat

(0.682770848274231, 12288, 1024, 12.0)

In [7]:
from app.services.compression_service import binary_quantize, hamming_distance

a_b = binary_quantize(vec_a)
b_b = binary_quantize(vec_b)
ham_b = hamming_distance(a_b, b_b)
mem_bin = 3072 // 8
ratio_bin = mem_orig / mem_bin
ham_b, mem_orig, mem_bin, ratio_bin

(866, 12288, 384, 32.0)

In [8]:
from app.services.compression_service import matryoshka_then_binary

a_mb = matryoshka_then_binary(vec_a, 256)
b_mb = matryoshka_then_binary(vec_b, 256)
ham_mb = hamming_distance(a_mb, b_mb)
mem_mb = 256 // 8
ratio_mb = mem_orig / mem_mb

rows = [
    {'strategy': 'matryoshka', 'metric': cos_m, 'memory_original': mem_orig, 'memory_compressed': mem_mat, 'compression_ratio': ratio_mat},
    {'strategy': 'binary', 'metric': ham_b, 'memory_original': mem_orig, 'memory_compressed': mem_bin, 'compression_ratio': ratio_bin},
    {'strategy': 'matryoshka_then_binary', 'metric': ham_mb, 'memory_original': mem_orig, 'memory_compressed': mem_mb, 'compression_ratio': ratio_mb},
]
rows

[{'strategy': 'matryoshka',
  'metric': 0.682770848274231,
  'memory_original': 12288,
  'memory_compressed': 1024,
  'compression_ratio': 12.0},
 {'strategy': 'binary',
  'metric': 866,
  'memory_original': 12288,
  'memory_compressed': 384,
  'compression_ratio': 32.0},
 {'strategy': 'matryoshka_then_binary',
  'metric': 76,
  'memory_original': 12288,
  'memory_compressed': 32,
  'compression_ratio': 384.0}]

## Analysis

### Q1: When reducing vector dimensions, what happens to magnitude and representation?

Truncating from 3072 to 256 dimensions does two things simultaneously:

**Magnitude drops.** The L2 norm of the sub-vector is always ≤ the original norm. For a unit-norm OpenAI vector, `||v[:256]||` is typically well below 1.0. This is why **L2 normalization after truncation is mandatory** — cosine similarity and binary quantization both assume a centered, unit-norm vector.

**Representation degrades gracefully** for Matryoshka-trained models. `text-embedding-3-large` uses Matryoshka Representation Learning (MRL), which front-loads the most semantically critical information into the first dimensions. The first 256 dims form a valid compact embedding on their own. In a standard (non-MRL) model, truncation discards information in an uncontrolled way with no guarantee the retained dims are the most meaningful.

---

### Q2: Why is L2 normalization required after truncation?

Two reasons, depending on what follows:

**For cosine similarity (Strategy A):** The formula `cos(θ) = a·b / (||a|| ||b||)` is technically magnitude-independent, so cosine similarity still works without normalization. However, L2 normalization is required for any downstream system that treats inner product as cosine similarity — including FAISS `IndexFlatIP` — and is a prerequisite if binary quantization follows.

**For binary quantization (Strategy C — the critical reason):** Binary quantization maps each value to its sign: `≥ 0 → 1`, `< 0 → 0`. The reliability of Hamming distance as a similarity proxy depends on the sign distribution being **balanced around zero** (~50% positive, ~50% negative). On a unit-norm vector this balance holds by construction. On an unnormalized truncated vector, the mean of the sub-dimensions is non-zero — more bits will be 1 (or 0) regardless of actual semantic similarity. This is exactly the bug that broke the e-commerce pipeline in the session: Matryoshka truncation followed immediately by binary quantization without the normalization step.

---

### Q3: How can binary quantization reduce memory footprint drastically while maintaining retrieval accuracy?

**Mechanism:** Each 32-bit float is replaced by 1 bit representing its sign. 32 floats pack into 1 uint8 byte via `numpy.packbits`. Memory reductions:

| Pipeline | Bytes per vector | Compression |
|:---|:---|:---|
| float32, 3072 dims (baseline) | 12,288 B | 1× |
| Strategy B — binary, 3072 dims | 384 B | **32×** |
| Strategy C — Matryoshka → binary, 256 dims | 32 B | **384×** |

**Why accuracy holds:** The sign of a normalized embedding dimension encodes the semantic direction along that latent axis. Two semantically similar vectors will agree in sign on most dimensions — Hamming distance counts disagreements, which inversely correlates with cosine similarity for L2-normalized inputs. Research from Cohere and Qdrant shows binary quantization of normalized vectors retains ~90–95% recall@10 versus full float32 search.

**The key precondition:** Vectors must be L2-normalized before binarization. Without this the sign distribution is biased, and the accuracy guarantee breaks — as demonstrated by the `broken_hamming` result in Cell 4.

---

### Q4: Which similarity metric should be used for binary vectors?

**Hamming distance** — not cosine similarity, not dot product.

`HD(a, b) = popcount(a XOR b)` — count the number of bit positions where `a` and `b` differ. In NumPy: `numpy.unpackbits(a ^ b).sum()`. Lower Hamming distance = more similar. Convert to a [0, 1] similarity score: `1 - HD / total_bits`.

**Why not cosine/dot product:** They require float multiplication. Applying them to packed uint8 arrays wastes every memory and speed advantage of packing. At the hardware level, Hamming distance maps to `POPCNT` instructions — orders of magnitude faster than float cosine at billion-scale retrieval.

| Strategy | Representation | Correct Metric |
|:---|:---|:---|
| A: Matryoshka | float32, L2-normalized | Cosine similarity |
| B: Direct binary | uint8 packed bits | Hamming distance |
| C: Chained pipeline | uint8 packed bits | Hamming distance |